<a href="https://colab.research.google.com/github/bcec0002/MMA3001-Project/blob/main/MMA3001_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MMA3001 Project: Training and evaluation notebook (pork rasher packaging defects)

Trains and evaluates YOLOv8 detectors for five packaging/product defect classes, then
scores defect severity from the predicted boxes. Part of the repository
[MMA3001-Project](https://github.com/bcec0002/MMA3001-Project). The severity code
lives in `src/`; this notebook does the GPU-dependent work (training, evaluation).

## Requirements
- **Runtime:** Google Colab, GPU (T4). Training on CPU is impractically slow.
- **Versions used:** Ultralytics 8.4.183, Python 3.13.15, torch 2.11.0+cu130
- **Roboflow API key:** stored in Colab Secrets under the name `API_KEY`. Never paste the key into a cell.
- **Google Drive:** mounted at `/content/drive`; outputs go to `MyDrive/MMA3001/runs/`.

## Data
- Roboflow project `pork-rasher-error-packaging`, workspace `hello-2aqe0`, **Version 4**, YOLOv8 format
- Splits: train 3,349 / valid 120 / test 80 images (684 / 14 / 12 contain no defects); 720 × 540 px
- Classes: loose-meat, packaging-error, twisted-meat, unsealed, wrinkle (wrinkle has no valid or test boxes)
- Licence and citation: [from the Roboflow dataset page]

## Cell order (code cells, counted after this header)
| Cell | Purpose | Notes |
|---|---|---|
| 1 | Install packages, mount Drive | Run every new session |
| 2 | Download dataset, print split counts | Expect 3349 / 120 / 80 |
| 3 | Train (parameterised) | ~20 to 45 min per run. Don't re-run unless retraining |
| 4 | Evaluate runs A, B, C on valid and test | Results already recorded; re-running creates duplicate folders |
| 5 | Clone this repo into Colab | Needs `src/` pushed to GitHub first |
| 6 | Severity functions and confidence sweep (valid) | Needs cells 1, 2, 5 |
| 7 | Final severity evaluation on test (`CONF = 0.25`) | Run once, after cell 6 |
| 8 | Scatter plot and five worst cases | Needs cell 7 |
| 9 | Worst-five image grid | Needs cell 8 |

## Experiments (30 epochs, batch 16, seed 0 for all)
| Run | Model | imgsz | Train time | Folder in Drive |
|---|---|---|---|---|
| A (selected) | yolov8n | 640 | ~40 s/epoch | `yolov8n_640_e30` |
| B | yolov8s | 640 | ~51 s/epoch | `yolov8s_640_e30` |
| C | yolov8n | 960 | 0.747 h total | `yolov8n_960_e30` |

Run C is evaluated at `imgsz=960`. Exact settings for each run: `results/args_*.yaml`.

## Selection protocol
Chosen on the **valid** split: C replaces A only if its valid packaging-error AP50 improves
by about 0.05 without hurting unsealed. It missed by a small margin (+0.048), so A is the
final model. Confidence threshold 0.25 chosen on valid severity error. Test results are
reported once, but were already visible when the models were compared.

## Caveats
Single seed per run; 80-image test set; rare classes have 3 to 4 test boxes, so their
metrics are unreliable; severity has no human-labelled ground truth.

## Outputs
Run folders in Drive; selected results copied to `results/` in the repo (settings, figures,
severity CSV, `run_outputs.md`).

In [ ]:
!pip install -q ultralytics roboflow

from google.colab import drive
drive.mount('/content/drive')
RUNS = "/content/drive/MyDrive/MMA3001/runs"

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("API_KEY"))
project = rf.workspace("hello-2aqe0").project("pork-rasher-error-packaging")
dataset = project.version(4).download("yolov8")
DATA_YAML = f"{dataset.location}/data.yaml"
print(DATA_YAML)
from pathlib import Path
for s in ["train", "valid", "test"]:
    print(s, len(list((Path(dataset.location) / s / "images").glob("*.jpg"))))

In [ ]:
from ultralytics import YOLO

# --- Experiment settings: the only things that change between runs ---
MODEL    = "yolov8n.pt"      # run A: yolov8n.pt | run B: yolov8s.pt | run C: yolov8n.pt
IMGSZ    = 640               # run A: 640        | run B: 640        | run C: 960
RUN_NAME = f"{MODEL.replace('.pt', '')}_{IMGSZ}_e30"
# ----------------------------------------------------------------------

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=IMGSZ,
    batch=16,
    seed=0,
    cache=False,         # stream images from disk rather than loading into RAM
    project=RUNS,
    name=RUN_NAME,
)

In [ ]:
# Evaluation of all three runs on the valid and test splits.
# Each run is evaluated at the image size it was trained at.
EVAL_RUNS = [
    ("yolov8n_640_e30", 640),   # run A (selected)
    ("yolov8s_640_e30", 640),   # run B
    ("yolov8n_960_e30", 960),   # run C
]

for run, imgsz in EVAL_RUNS:
    mdl = YOLO(f"{RUNS}/{run}/weights/best.pt")
    for split, tag in [("val", "valid"), ("test", "test")]:
        m = mdl.val(data=DATA_YAML, split=split, imgsz=imgsz,
                    project=RUNS, name=f"{run}_{tag}")
        print(f"\n{run} | {tag.upper()} mAP50: {m.box.map50:.3f}  "
              f"mAP50-95: {m.box.map:.3f}  P: {m.box.mp:.3f}  R: {m.box.mr:.3f}")
        for i, c in enumerate(m.box.ap_class_index):
            print("  ", m.names[c], "AP50 =", round(float(m.box.ap50[i]), 3))
        print("   speed (ms/img):", m.speed)
    mdl.info()   # parameters and GFLOPs (reported at the default 640 input, even for run C)

In [ ]:
!git clone https://github.com/bcec0002/MMA3001-Project.git /content/repo || git -C /content/repo pull

In [ ]:
from ultralytics import YOLO
import sys
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.append("/content/repo")
from src.severity import tray_severity, DEFAULT_WEIGHTS
print(DEFAULT_WEIGHTS)
from src.labels import read_labels

model = YOLO(f"{RUNS}/yolov8n_640_e30/weights/best.pt")   # selected model (A)
names = model.names                                         # {0: 'loose-meat', ...}
ROOT_DS = Path(dataset.location)

def severity_table(split, conf):
    """Per-image severity from ground-truth boxes and from predicted boxes."""
    rows = []
    for r in model.predict(source=str(ROOT_DS / split / "images"), conf=conf,
                           imgsz=640, stream=True, verbose=False):
        stem = Path(r.path).stem
        gt = [(names[c], w, h) for c, _, _, w, h in
              read_labels(ROOT_DS / split / "labels" / f"{stem}.txt")]
        pr = [(names[int(c)], w, h) for c, (_, _, w, h) in
              zip(r.boxes.cls.tolist(), r.boxes.xywhn.tolist())]
        rows.append({"image": stem, "n_gt": len(gt), "n_pred": len(pr),
                     "sev_gt": tray_severity(gt), "sev_pred": tray_severity(pr)})
    return pd.DataFrame(rows)

# 1) Choose the confidence threshold on the VALID split (never on test)
print("conf | valid MAE | valid Spearman")
for conf in [0.1, 0.25, 0.4, 0.5]:
    d = severity_table("valid", conf)
    mae = (d.sev_pred - d.sev_gt).abs().mean()
    print(f"{conf:.2f} | {mae:.4f} | {d.sev_gt.corr(d.sev_pred, method='spearman'):.3f}")

In [ ]:
CONF = 0.25  # chosen on valid in the sweep above

dv = severity_table("valid", CONF)
dt = severity_table("test", CONF)

# Naive baseline: always predict the mean valid ground-truth severity
base_mae = (dt.sev_gt - dv.sev_gt.mean()).abs().mean()
mae = (dt.sev_pred - dt.sev_gt).abs().mean()
rho = dt.sev_gt.corr(dt.sev_pred, method="spearman")

# Bootstrap 95% interval for MAE (resample test images)
rng = np.random.default_rng(0)
err = (dt.sev_pred - dt.sev_gt).abs().to_numpy()
boot = [rng.choice(err, len(err)).mean() for _ in range(2000)]
print(f"TEST MAE {mae:.4f} (95% CI {np.percentile(boot, 2.5):.4f}-{np.percentile(boot, 97.5):.4f}), "
      f"naive baseline MAE {base_mae:.4f}, Spearman {rho:.3f}")

# Pass/fail agreement across thresholds (fail = severity >= threshold)
print("\nthreshold | accuracy | failing trays caught | false fails")
for t in [0.005, 0.01, 0.02, 0.05, 0.1]:
    f_gt, f_pr = dt.sev_gt >= t, dt.sev_pred >= t
    acc = (f_gt == f_pr).mean()
    caught = (f_gt & f_pr).sum() / max(f_gt.sum(), 1)
    false_fail = (~f_gt & f_pr).sum()
    print(f"{t:.3f} | {acc:.2f} | {caught:.2f} ({int(f_gt.sum())} true fails) | {false_fail}")

dt.to_csv(f"{RUNS}/severity_test_yolov8n_640.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(dt.sev_gt, dt.sev_pred, s=15, alpha=0.7)
lim = max(dt.sev_gt.max(), dt.sev_pred.max()) * 1.05
ax.plot([0, lim], [0, lim], "k--", lw=1, label="perfect agreement")
ax.set_xlabel("Severity from ground-truth boxes")
ax.set_ylabel("Severity from predicted boxes")
ax.legend()
plt.tight_layout()
plt.savefig(f"{RUNS}/severity_scatter_test.png", dpi=200)
plt.show()

dt["abs_err"] = (dt.sev_pred - dt.sev_gt).abs()
print(dt.sort_values("abs_err", ascending=False).head(5).to_string(index=False))

In [ ]:
import cv2
import matplotlib.pyplot as plt

def draw_box(img, xc, yc, bw, bh, text, colour):
    h, w = img.shape[:2]
    x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
    x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
    cv2.rectangle(img, (x1, y1), (x2, y2), colour, 2)
    cv2.putText(img, text, (x1, max(y1 - 5, 12)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, colour, 1)

worst = dt.sort_values("abs_err", ascending=False).head(5)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (_, row) in zip(axes.ravel(), worst.iterrows()):
    p = ROOT_DS / "test" / "images" / f"{row.image}.jpg"
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    for c, xc, yc, bw, bh in read_labels(ROOT_DS / "test" / "labels" / f"{row.image}.txt"):
        draw_box(img, xc, yc, bw, bh, "GT " + names[c], (0, 200, 0))
    r = model.predict(str(p), conf=0.25, imgsz=640, verbose=False)[0]
    for c, (xc, yc, bw, bh), s in zip(r.boxes.cls.tolist(),
                                      r.boxes.xywhn.tolist(), r.boxes.conf.tolist()):
        draw_box(img, xc, yc, bw, bh, f"{names[int(c)]} {s:.2f}", (255, 0, 0))
    ax.imshow(img)
    ax.set_title(f"GT {row.sev_gt:.3f} / pred {row.sev_pred:.3f}", fontsize=9)
    ax.axis("off")
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.savefig(f"{RUNS}/severity_worst5_test.png", dpi=200)
plt.show()